# Microcinética: conferir contra a tese

Lê uma rodada gravada pelo `run.ipynb` e compara com os arquivos do ESI, que são a
fonte da verdade:

| arquivo do ESI | o que traz |
|---|---|
| `tables/ch2.microkineticsLibrary.csv` | as 136 reações da biblioteca nas quatro condições padrão |
| `tables/ch2.microkineticsReported.csv` | as reações publicadas nas condições de preparo de cada uma |
| `tables/ch2.microkineticsReportedStandard.csv` | as mesmas nas quatro condições padrão |

**Duas perguntas diferentes**, e o relatório separa as duas:

- **reprodução** — se você partiu das energias do ESI, os números têm de sair iguais,
  porque a semente e o esquema de perturbação são os mesmos. Aqui a tolerância é apertada.
- **concordância** — se você trouxe energias próprias, o que se compara é o **produto
  previsto** (o lado do corte de 0,5) e a **margem** (a distância a 0,5). O p_MAP mede
  o erro do cálculo, não o acerto experimental.



## 1. Onde está a rodada

In [ ]:
from pathlib import Path
import sys

import pandas as pd

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from src import compare, runner

ROOT = HERE.parents[0]
ESI = ROOT / "tables"   # as tabelas do material suplementar

RUN_DIR = HERE / "exports/rodada-exemplo"    # a pasta que o run.ipynb gravou
CONDITIONS = ("A1", "A2", "B1", "B2")

# Tolerâncias. Apertadas de propósito: partindo das energias do ESI, com a mesma
# semente, a diferença tem de ser zero. Afrouxe se você trouxe energias próprias.
P_MAP_TOLERANCE = 0.05      # 5 pontos percentuais na fração
MEDIAN_TOLERANCE = 1.0      # 1 ponto percentual nas medianas de BP, MAP e BIS

frames = runner.load_run_frames(RUN_DIR, CONDITIONS)
print(f"{len(frames)} condições em {RUN_DIR}")
for condition, frame in frames.items():
    print(f"  {condition}: {len(frame)} reações")

## 2. Confronto com o ESI

In [ ]:
comparacoes = {}

biblioteca = compare.compare_library(frames, ESI,
                                     p_map_tolerance=P_MAP_TOLERANCE,
                                     median_tolerance=MEDIAN_TOLERANCE,
                                     conditions=CONDITIONS)
if not biblioteca.empty:
    comparacoes["biblioteca (condições padrão)"] = biblioteca

publicadas = compare.compare_reported(frames, ESI, conditions=CONDITIONS)
if not publicadas.empty:
    comparacoes["publicadas (condições padrão)"] = publicadas

resumo = compare.report(comparacoes, title="Conferência contra o ESI")

## 3. Onde divergiu

As linhas que não conferem, com o que cada lado previu.

In [ ]:
if biblioteca.empty:
    print("sem linhas da biblioteca para comparar")
else:
    divergentes = biblioteca[biblioteca["verdict"] != "confere"]
    print(f"{len(divergentes)} de {len(biblioteca)} linhas divergem\n")
    if not divergentes.empty:
        colunas = ["reaction", "condition", "verdict", "p_MAP_rodada", "p_MAP_ESI",
                   "produto_rodada", "produto_ESI", "margem_ESI"]
        display(divergentes[colunas].round(3))

## 4. Produto previsto: rodada × ESI

O eixo x é o p_MAP da tese, o y é o da rodada; a cor é a margem (a distância ao corte
de 0,5) do lado do ESI. Pontos na diagonal são reprodução exata. Fundo dividido pelo
produto previsto: a linha do corte é o 0,5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from src import colors

MAPA, BIS = colors.PRODUCT["MAP"], colors.PRODUCT["BIS"]

if not biblioteca.empty:
    blocos = [("biblioteca", biblioteca)]
    if not publicadas.empty:
        blocos.append(("publicadas", publicadas))

    fig, axes = plt.subplots(1, len(blocos), figsize=(3.6 * len(blocos), 3.4), squeeze=False)
    for ax, (nome, dados) in zip(axes[0], blocos):
        valido = dados.dropna(subset=["p_MAP_rodada", "p_MAP_ESI"])
        ax.axhspan(0, 0.5, color=BIS, alpha=0.07)
        ax.axhspan(0.5, 1, color=MAPA, alpha=0.07)
        ax.axvline(0.5, color="0.6", lw=0.8, ls="--")
        ax.axhline(0.5, color="0.6", lw=0.8, ls="--")
        ax.plot([0, 1], [0, 1], color="0.75", lw=0.8, zorder=1)
        ok = valido["verdict"] == "confere"
        ax.scatter(valido.loc[ok, "p_MAP_ESI"], valido.loc[ok, "p_MAP_rodada"],
                   s=22, color="0.35", edgecolor="white", linewidth=0.4, zorder=3, label="confere")
        if (~ok).any():
            ax.scatter(valido.loc[~ok, "p_MAP_ESI"], valido.loc[~ok, "p_MAP_rodada"],
                       s=34, facecolor="none", edgecolor="#CC3311", linewidth=1.0,
                       zorder=4, label="diverge")
        ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel("$p_{MAP}$ da tese")
        ax.set_title(nome, fontsize=9)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)
    axes[0][0].set_ylabel("$p_{MAP}$ da rodada")
    axes[0][-1].legend(fontsize=7, frameon=False, loc="lower right")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "conferencia-pmap.png", dpi=200)
    plt.show()
    print(f"figura: {RUN_DIR / 'conferencia-pmap.png'}")

## 5. Gravar o relatório

Um CSV por confronto, ao lado da rodada.

In [ ]:
for nome, tabela in comparacoes.items():
    destino = RUN_DIR / f"conferencia-{nome.split()[0]}.csv"
    tabela.to_csv(destino, index=False)
    print(f"{destino}  ({len(tabela)} linhas)")

print("\nLeitura do resultado:")
print("  'confere'                    -> produto previsto igual e números dentro da tolerância")
print("  'produto confere, números divergem' -> o produto majoritário bate, mas as frações não")
print("  'produto divergente'         -> a rodada prevê o outro produto")